# Vague Benchmark Analysis

Cells in order:
1. Setup — imports, llm_fn configuration
2. LongBench — compare_all on qasper, F1 bar chart
3. Needle heatmap — run_needle_sweep, seaborn heatmap
4. Compression analysis — compression_ratio vs F1 scatter
5. Summary table — markdown table of all EvalResult fields

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import os, sys, random
import numpy as np
from dotenv import load_dotenv
load_dotenv()

# Ensure project root is importable when running from notebooks/
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Determinism
random.seed(42)
np.random.seed(42)

# Results directory
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'benchmarks', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
CACHE_DIR = os.path.join(PROJECT_ROOT, '.cache')
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Plug in your LLM here ──────────────────────────────────────────────────
# llm_fn must be Callable[[str], str]  (prompt in, completion out).
#
# Example 1 — OpenAI
# import openai
# client = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])
# def llm_fn(prompt: str) -> str:
#     r = client.chat.completions.create(
#         model='gpt-4o-mini',
#         messages=[{'role': 'user', 'content': prompt}],
#         max_tokens=256,
#     )
#     return r.choices[0].message.content
#
# Example 2 — Anthropic
import anthropic
client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
def llm_fn(prompt: str) -> str:
     r = client.messages.create(
         model='claude-3-haiku-20240307',
         max_tokens=256,
         messages=[{'role': 'user', 'content': prompt}],
     )
     return r.content[0].text
#
# Example 3 — local Ollama
# import requests
# def llm_fn(prompt: str) -> str:
#     r = requests.post('http://localhost:11434/api/generate',
#                       json={'model': 'llama3', 'prompt': prompt, 'stream': False})
#     return r.json()['response']

print('Setup complete.')
print('RESULTS_DIR:', RESULTS_DIR)

In [ ]:
# ── Cell 2: LongBench — compare_all on qasper ──────────────────────────────
from benchmarks.longbench import LongBenchEval, EvalResult

N_SAMPLES = 50   # increase to 200 for publication-quality results

evaluator = LongBenchEval(llm_fn=llm_fn, cache_dir=CACHE_DIR)
results: list[EvalResult] = evaluator.compare_all(task='qasper', n_samples=N_SAMPLES)

for r in results:
    print(f'{r.method:15s}  F1={r.f1_score:.3f}  '
          f'tokens={r.avg_input_tokens}  '
          f'cr={r.compression_ratio:.2f}  '
          f'latency={r.latency_ms:.1f}ms')

# Bar chart: F1 vs method, annotated with avg token counts
methods      = [r.method for r in results]
f1_scores    = [r.f1_score for r in results]
token_counts = [r.avg_input_tokens for r in results]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    methods, f1_scores,
    color=['#4C72B0', '#DD8452', '#55A868'],
    edgecolor='white',
)
ax.set_xlabel('Method', fontsize=12)
ax.set_ylabel('Token-level F1', fontsize=12)
ax.set_title('LongBench — qasper: F1 by Method', fontsize=13)
top = max(f1_scores) if f1_scores else 1.0
ax.set_ylim(0, min(1.05, top * 1.35 + 0.05))

for bar, tok in zip(bars, token_counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{tok} tok',
        ha='center', va='bottom', fontsize=9, color='#333333',
    )

plt.tight_layout()
out_path = os.path.join(RESULTS_DIR, 'longbench_qasper_f1.png')
plt.savefig(out_path, dpi=150)
plt.show()
print('Saved:', out_path)

In [ ]:
# ── Cell 3: Needle heatmap ─────────────────────────────────────────────────
from benchmarks.needle import run_needle_sweep

CONTEXT_LENGTHS = [512, 1024, 2048, 4096, 8192]
POSITIONS       = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]

# Pass llm_fn=None to use the always-found mock (structural test).
# Replace with your real llm_fn for actual recall measurements.
sweep_df = run_needle_sweep(
    context_lengths=CONTEXT_LENGTHS,
    positions=POSITIONS,
    llm_fn=llm_fn,
)
print(sweep_df.to_string(index=False))

# Pivot for heatmap (context_length x position)
pivot = sweep_df.pivot(index='context_length', columns='position', values='found_rate')
pivot = pivot.sort_index(ascending=False)   # longest context at top

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(
    pivot,
    ax=ax,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=0.0,
    vmax=1.0,
    linewidths=0.5,
    cbar_kws={'label': 'Found rate'},
)
ax.set_title('Needle-in-a-Haystack: Found Rate (BeliefMemory)', fontsize=13)
ax.set_xlabel('Needle position (0=start, 1=end)', fontsize=11)
ax.set_ylabel('Context length (tokens)', fontsize=11)

plt.tight_layout()
out_path = os.path.join(RESULTS_DIR, 'needle_heatmap.png')
plt.savefig(out_path, dpi=150)
plt.show()
print('Saved:', out_path)

In [ ]:
# ── Cell 4: Compression analysis — scatter (compression_ratio vs F1) ───────
N_COMPONENTS_GRID = [8, 16, 32, 64, 128]
TASK = 'qasper'
N_SAMPLES_COMP = 30   # keep quick; increase for statistical confidence

comp_results = []
for nc in N_COMPONENTS_GRID:
    print(f'Running vague n_components={nc} ...')
    r = evaluator.run(
        task=TASK, method='vague',
        n_components=nc, n_samples=N_SAMPLES_COMP,
    )
    comp_results.append(r)
    print(f'  F1={r.f1_score:.3f}  cr={r.compression_ratio:.3f}  tokens={r.avg_input_tokens}')

crs    = [r.compression_ratio for r in comp_results]
f1s    = [r.f1_score for r in comp_results]
labels = [str(nc) for nc in N_COMPONENTS_GRID]

fig, ax = plt.subplots(figsize=(6, 4))
sc = ax.scatter(
    crs, f1s,
    c=N_COMPONENTS_GRID, cmap='viridis',
    s=100, zorder=3, edgecolors='white',
)
for x, y, lbl in zip(crs, f1s, labels):
    ax.annotate(
        f'nc={lbl}', (x, y),
        textcoords='offset points', xytext=(6, 4),
        fontsize=8, color='#333333',
    )

plt.colorbar(sc, ax=ax, label='n_components')
ax.set_xlabel('Compression ratio', fontsize=11)
ax.set_ylabel('Token-level F1', fontsize=11)
ax.set_title(f'Vague: compression_ratio vs F1 ({TASK})', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_path = os.path.join(RESULTS_DIR, 'compression_vs_f1.png')
plt.savefig(out_path, dpi=150)
plt.show()
print('Saved:', out_path)

## Cell 5: Summary Table — All EvalResult Fields

Run the code cell below to generate the markdown table from all collected results.

Fields: `task`, `method`, `f1_score`, `avg_input_tokens`, `compression_ratio`, `latency_ms`, `n_samples`

In [ ]:
# ── Cell 5: Summary table ──────────────────────────────────────────────────
from IPython.display import Markdown, display
import dataclasses

all_results = results + comp_results

header = '| task | method | f1_score | avg_input_tokens | compression_ratio | latency_ms | n_samples |'
sep    = '|------|--------|----------|-----------------|-------------------|------------|-----------|'
rows   = []
for r in all_results:
    rows.append(
        f'| {r.task} | {r.method} | {r.f1_score:.3f} '
        f'| {r.avg_input_tokens} '
        f'| {r.compression_ratio:.3f} '
        f'| {r.latency_ms:.1f} '
        f'| {r.n_samples} |'
    )

table = '\n'.join([header, sep] + rows)
display(Markdown(table))
print(table)